# A100 SU(3) pure-gauge + $T_1^{+-}$ spectroscopy pilot

**Run without editing:** choose **Runtime → Change runtime type → A100 GPU**, then run the single code cell.

The notebook generates Wilson-action ensembles, checkpoints automatically, and measures plaquettes, Wilson loops, spatial torelons, and $T_1^{+-}$ correlators at $\Gamma,X,M,R$.

In [ ]:
#!/usr/bin/env python3
"""
A100 SU(3) PURE-GAUGE PILOT
===========================
Single-file, Colab-ready isotropic Euclidean Wilson-action simulation.

Purpose
-------
1. Generate pure-SU(3) gauge ensembles on an NVIDIA A100.
2. Measure plaquette histories and Wilson loops.
3. Measure spatial torelon correlators as a string-tension proxy.
4. Measure APE-smeared C-odd T1^{+-} plaquette operators at
   Gamma, X, M, R and their Euclidean-time correlators.
5. Save resumable checkpoints, raw arrays, JSON summaries, and plots.

Scientific scope
----------------
This is a numerical Euclidean pilot, not an exact certificate and not yet an
anisotropic Hamiltonian-limit calculation. Its first job is to validate the
GPU pipeline and test the predicted momentum ordering and band shape.

Default run
-----------
Designed for a single A100 in Google Colab. No editing is required.
Outputs are written under /content/A100_SU3_T1PM_RUN when /content exists,
otherwise under the current working directory.

Environment-variable overrides are supported for debugging, e.g.
  GLUE_L=4 GLUE_LT=8 GLUE_THERM=4 GLUE_NMEAS=3 GLUE_GAP=1 python script.py
"""

from __future__ import annotations

import json
import math
import os
import platform
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Tuple

# Force 64-bit before importing jax.numpy.
os.environ.setdefault("JAX_ENABLE_X64", "True")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

try:
    import jax
    import jax.numpy as jnp
    from jax import lax, random
except Exception as exc:  # pragma: no cover
    raise RuntimeError(
        "JAX is required. In Colab, select Runtime > Change runtime type > A100 GPU."
    ) from exc

import matplotlib.pyplot as plt
import numpy as np

jax.config.update("jax_enable_x64", True)


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, default))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, default))


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


@dataclass
class Config:
    # Even spatial size is required for X/M/R momenta.
    L: int = env_int("GLUE_L", 8)
    LT: int = env_int("GLUE_LT", 16)
    beta: float = env_float("GLUE_BETA", 5.50)
    seed: int = env_int("GLUE_SEED", 20260614)

    # Pilot statistics. Increase after validation.
    therm_sweeps: int = env_int("GLUE_THERM", 120)
    n_measurements: int = env_int("GLUE_NMEAS", 96)
    sweeps_between: int = env_int("GLUE_GAP", 3)

    # Metropolis proposal and adaptation.
    epsilon: float = env_float("GLUE_EPS", 0.24)
    target_accept: float = env_float("GLUE_TARGET_ACCEPT", 0.52)
    adapt_every: int = env_int("GLUE_ADAPT_EVERY", 10)

    # Measurement controls.
    ape_steps: int = env_int("GLUE_APE_STEPS", 5)
    ape_alpha: float = env_float("GLUE_APE_ALPHA", 0.45)
    wilson_every: int = env_int("GLUE_WILSON_EVERY", 4)
    checkpoint_every: int = env_int("GLUE_CHECKPOINT_EVERY", 8)
    bootstrap_samples: int = env_int("GLUE_BOOTSTRAP", 300)

    # Starts from a Haar-random field. Set false only for controlled debugging.
    hot_start: bool = env_bool("GLUE_HOT_START", True)
    allow_cpu: bool = env_bool("GLUE_ALLOW_CPU", False)

    @property
    def shape4(self) -> Tuple[int, int, int, int]:
        return (self.LT, self.L, self.L, self.L)

    @property
    def volume(self) -> int:
        return self.LT * self.L**3


CFG = Config()
if CFG.L % 2:
    raise ValueError("GLUE_L must be even so that X, M, and R are lattice momenta.")

ROOT = Path("/content/A100_SU3_T1PM_RUN") if Path("/content").exists() else Path.cwd() / "A100_SU3_T1PM_RUN"
ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT = ROOT / "checkpoint_latest.npz"
RAW_NPZ = ROOT / "raw_measurements.npz"
SUMMARY_JSON = ROOT / "summary.json"
LOG_TXT = ROOT / "run.log"


class Tee:
    def __init__(self, path: Path):
        self.stdout = sys.stdout
        self.file = path.open("a", encoding="utf-8")

    def write(self, text: str) -> None:
        self.stdout.write(text)
        self.file.write(text)
        self.file.flush()

    def flush(self) -> None:
        self.stdout.flush()
        self.file.flush()


sys.stdout = Tee(LOG_TXT)


# -----------------------------------------------------------------------------
# Linear-algebra helpers
# -----------------------------------------------------------------------------

CDTYPE = jnp.complex128
RDTYPE = jnp.float64
I3 = jnp.eye(3, dtype=CDTYPE)
AXES4 = (0, 1, 2, 3)


def dagger(a: jax.Array) -> jax.Array:
    return jnp.swapaxes(jnp.conj(a), -1, -2)


def trace3(a: jax.Array) -> jax.Array:
    return jnp.trace(a, axis1=-2, axis2=-1)


def shift_field(a: jax.Array, offset: Tuple[int, int, int, int]) -> jax.Array:
    # output[x] = input[x + offset]
    return jnp.roll(a, shift=tuple(-int(v) for v in offset), axis=AXES4)


def unit_offset(mu: int, sign: int = 1) -> Tuple[int, int, int, int]:
    out = [0, 0, 0, 0]
    out[mu] = sign
    return tuple(out)


def add_offset(*offsets: Tuple[int, int, int, int]) -> Tuple[int, int, int, int]:
    return tuple(sum(o[i] for o in offsets) for i in range(4))


def link_at(U: jax.Array, mu: int, offset: Tuple[int, int, int, int]) -> jax.Array:
    return shift_field(U[..., mu, :, :], offset)


def project_to_su3_batch(M: jax.Array) -> jax.Array:
    """Polar/SVD projection of a batch of nonsingular 3x3 matrices to SU(3)."""
    u, _, vh = jnp.linalg.svd(M, full_matrices=False)
    q = u @ vh
    detq = jnp.linalg.det(q)
    # Multiplying the last column by conj(det q) sets det exactly to one.
    q = q.at[..., :, 2].multiply(jnp.conj(detq)[..., None])
    return q


def haar_su3(key: jax.Array, leading_shape: Tuple[int, ...]) -> jax.Array:
    """Batched Haar SU(3) via complex Ginibre QR and determinant correction."""
    key_r, key_i = random.split(key)
    z = random.normal(key_r, leading_shape + (3, 3), dtype=RDTYPE)
    z = z + 1j * random.normal(key_i, leading_shape + (3, 3), dtype=RDTYPE)
    q, r = jnp.linalg.qr(z)
    diag = jnp.diagonal(r, axis1=-2, axis2=-1)
    phase = diag / jnp.where(jnp.abs(diag) > 0, jnp.abs(diag), 1.0)
    q = q * jnp.conj(phase)[..., None, :]
    detq = jnp.linalg.det(q)
    q = q.at[..., :, 2].multiply(jnp.conj(detq)[..., None])
    return q.astype(CDTYPE)


# -----------------------------------------------------------------------------
# Wilson action and updates
# -----------------------------------------------------------------------------

def staple(U: jax.Array, mu: int, spatial_only: bool = False) -> jax.Array:
    """Sum of the six 4D staples, or four spatial staples for APE smearing."""
    result = jnp.zeros(U.shape[:4] + (3, 3), dtype=CDTYPE)
    directions = (1, 2, 3) if spatial_only else (0, 1, 2, 3)
    emu = unit_offset(mu, +1)
    for nu in directions:
        if nu == mu:
            continue
        enu = unit_offset(nu, +1)
        mnu = unit_offset(nu, -1)
        # Forward plaquette: U_mu(x) S_f(x)
        sf = (
            link_at(U, nu, emu)
            @ dagger(link_at(U, mu, enu))
            @ dagger(link_at(U, nu, (0, 0, 0, 0)))
        )
        # Backward plaquette, cyclically ordered after U_mu(x).
        sb = (
            dagger(link_at(U, nu, add_offset(emu, mnu)))
            @ dagger(link_at(U, mu, mnu))
            @ link_at(U, nu, mnu)
        )
        result = result + sf + sb
    return result


def su2_proposal(key: jax.Array, shape4: Tuple[int, int, int, int], epsilon: float, subgroup: int) -> jax.Array:
    """Near-identity random SU(2) embedded in one of the three SU(3) subgroups."""
    key_axis, key_theta = random.split(key)
    axis = random.normal(key_axis, shape4 + (3,), dtype=RDTYPE)
    axis = axis / jnp.maximum(jnp.linalg.norm(axis, axis=-1, keepdims=True), 1e-15)
    theta = random.uniform(key_theta, shape4, minval=-epsilon, maxval=epsilon, dtype=RDTYPE)
    s = jnp.sin(theta)
    a0 = jnp.cos(theta)
    a1, a2, a3 = axis[..., 0] * s, axis[..., 1] * s, axis[..., 2] * s

    r00 = a0 + 1j * a3
    r01 = a2 + 1j * a1
    r10 = -a2 + 1j * a1
    r11 = a0 - 1j * a3

    R = jnp.broadcast_to(I3, shape4 + (3, 3))
    pair = ((0, 1), (0, 2), (1, 2))[subgroup]
    i, j = pair
    R = R.at[..., i, i].set(r00)
    R = R.at[..., i, j].set(r01)
    R = R.at[..., j, i].set(r10)
    R = R.at[..., j, j].set(r11)
    return R


coords = jnp.indices(CFG.shape4)
PARITY_FIELD = (jnp.sum(coords, axis=0) & 1).astype(jnp.int32)


def update_substep_impl(
    U: jax.Array,
    key: jax.Array,
    beta: float,
    epsilon: float,
    mu: int,
    parity: int,
    subgroup: int,
) -> Tuple[jax.Array, jax.Array, jax.Array]:
    key_prop, key_acc, key_next = random.split(key, 3)
    S = staple(U, mu, spatial_only=False)
    old = U[..., mu, :, :]
    R = su2_proposal(key_prop, CFG.shape4, epsilon, subgroup)
    proposal = R @ old
    old_score = jnp.real(trace3(old @ S))
    new_score = jnp.real(trace3(proposal @ S))
    delta = (beta / 3.0) * (new_score - old_score)
    logu = jnp.log(random.uniform(key_acc, CFG.shape4, minval=1e-300, maxval=1.0, dtype=RDTYPE))
    active = PARITY_FIELD == parity
    accept = active & (logu < delta)
    updated = jnp.where(accept[..., None, None], proposal, old)
    U = U.at[..., mu, :, :].set(updated)
    return U, key_next, jnp.sum(accept)


# Static arguments produce one optimized kernel for each direction/parity/subgroup.
update_substep = jax.jit(update_substep_impl, static_argnames=("mu", "parity", "subgroup"))


def sweep(U: jax.Array, key: jax.Array, beta: float, epsilon: float) -> Tuple[jax.Array, jax.Array, float]:
    accepted = 0.0
    active_total = 0
    for mu in range(4):
        for parity in (0, 1):
            for subgroup in range(3):
                U, key, nacc = update_substep(U, key, beta, epsilon, mu, parity, subgroup)
                accepted += float(nacc)
                active_total += CFG.volume // 2
    return U, key, accepted / active_total


# -----------------------------------------------------------------------------
# Measurements
# -----------------------------------------------------------------------------

def plaquette_matrix(U: jax.Array, mu: int, nu: int) -> jax.Array:
    emu = unit_offset(mu, +1)
    enu = unit_offset(nu, +1)
    return (
        link_at(U, mu, (0, 0, 0, 0))
        @ link_at(U, nu, emu)
        @ dagger(link_at(U, mu, enu))
        @ dagger(link_at(U, nu, (0, 0, 0, 0)))
    )


@jax.jit
def mean_plaquette(U: jax.Array) -> jax.Array:
    total = 0.0
    count = 0
    for mu in range(4):
        for nu in range(mu + 1, 4):
            total = total + jnp.mean(jnp.real(trace3(plaquette_matrix(U, mu, nu))) / 3.0)
            count += 1
    return total / count


def ape_smear_impl(U: jax.Array, alpha: float, steps: int) -> jax.Array:
    """APE smear spatial links only; temporal links remain unchanged."""
    out = U
    for _ in range(steps):
        new_out = out
        for mu in (1, 2, 3):
            st = staple(out, mu, spatial_only=True)
            candidate = (1.0 - alpha) * out[..., mu, :, :] + (alpha / 4.0) * st
            new_out = new_out.at[..., mu, :, :].set(project_to_su3_batch(candidate))
        out = new_out
    return out


ape_smear = jax.jit(ape_smear_impl, static_argnames=("steps",))


def path_link(U: jax.Array, direction: int, offset: List[int], sign: int) -> Tuple[jax.Array, List[int]]:
    if sign > 0:
        link = link_at(U, direction, tuple(offset))
        offset = offset.copy()
        offset[direction] += 1
        return link, offset
    offset = offset.copy()
    offset[direction] -= 1
    link = dagger(link_at(U, direction, tuple(offset)))
    return link, offset


def rectangular_loop_impl(U: jax.Array, mu: int, nu: int, r: int, t: int) -> jax.Array:
    M = jnp.broadcast_to(I3, U.shape[:4] + (3, 3))
    offset = [0, 0, 0, 0]
    for _ in range(r):
        lk, offset = path_link(U, mu, offset, +1)
        M = M @ lk
    for _ in range(t):
        lk, offset = path_link(U, nu, offset, +1)
        M = M @ lk
    for _ in range(r):
        lk, offset = path_link(U, mu, offset, -1)
        M = M @ lk
    for _ in range(t):
        lk, offset = path_link(U, nu, offset, -1)
        M = M @ lk
    return jnp.mean(jnp.real(trace3(M)) / 3.0)


rectangular_loop = jax.jit(rectangular_loop_impl, static_argnames=("mu", "nu", "r", "t"))


def polyakov_field_impl(U: jax.Array, mu: int, length: int) -> jax.Array:
    M = jnp.broadcast_to(I3, U.shape[:4] + (3, 3))
    offset = [0, 0, 0, 0]
    for _ in range(length):
        M = M @ link_at(U, mu, tuple(offset))
        offset[mu] += 1
    return trace3(M) / 3.0


polyakov_field = jax.jit(polyakov_field_impl, static_argnames=("mu", "length"))


# Momentum phases for Gamma, X, M, R. Spatial axes are x,y,z = lattice axes 1,2,3.
MOMENTA = {
    "G": (0, 0, 0),
    "X": (1, 0, 0),
    "M": (1, 1, 0),
    "R": (1, 1, 1),
}
MOMENTUM_NAMES = tuple(MOMENTA.keys())
xyz = np.indices((CFG.L, CFG.L, CFG.L))
phase_np = []
for name in MOMENTUM_NAMES:
    bits = MOMENTA[name]
    parity = bits[0] * xyz[0] + bits[1] * xyz[1] + bits[2] * xyz[2]
    phase_np.append(np.exp(-1j * np.pi * parity))
PHASES = jnp.asarray(np.stack(phase_np, axis=0), dtype=CDTYPE)


@jax.jit
def t1pm_momentum_operators(U_smeared: jax.Array) -> jax.Array:
    """
    C-odd axial-vector plaquette operator:
      B_x = Im Tr P_yz, B_y = -Im Tr P_xz, B_z = Im Tr P_xy.
    Returns O[k, t, component], normalized by sqrt(spatial volume).
    """
    p_yz = jnp.imag(trace3(plaquette_matrix(U_smeared, 2, 3)))
    p_xz = jnp.imag(trace3(plaquette_matrix(U_smeared, 1, 3)))
    p_xy = jnp.imag(trace3(plaquette_matrix(U_smeared, 1, 2)))
    B = jnp.stack((p_yz, -p_xz, p_xy), axis=-1)  # [t,x,y,z,c]
    O = jnp.einsum("kxyz,txyzc->ktc", PHASES, B, optimize=True)
    return O / math.sqrt(CFG.L**3)


@jax.jit
def spatial_torelon_operators(U_smeared: jax.Array) -> jax.Array:
    """Returns P[t, spatial_direction] averaged over all spatial starting sites."""
    vals = []
    for mu in (1, 2, 3):
        field = polyakov_field(U_smeared, mu, CFG.L)
        vals.append(jnp.mean(field, axis=(1, 2, 3)))
    return jnp.stack(vals, axis=-1)


# -----------------------------------------------------------------------------
# Statistics and post-processing
# -----------------------------------------------------------------------------

def periodic_correlator(data: np.ndarray, subtract_mean: bool = True) -> np.ndarray:
    """Average over configurations, time origins, and trailing operator components."""
    z = np.asarray(data)
    if subtract_mean:
        z = z - z.mean(axis=(0, 1), keepdims=True)
    lt = z.shape[1]
    out = np.empty(lt, dtype=np.float64)
    for tau in range(lt):
        shifted = np.roll(z, -tau, axis=1)
        out[tau] = np.real(np.mean(shifted * np.conj(z)))
    return out


def effective_mass_log(c: np.ndarray) -> np.ndarray:
    c = np.asarray(c, dtype=float)
    out = np.full(max(len(c) - 1, 0), np.nan)
    valid = (c[:-1] > 0) & (c[1:] > 0)
    out[valid] = np.log(c[:-1][valid] / c[1:][valid])
    return out


def effective_mass_cosh(c: np.ndarray) -> np.ndarray:
    c = np.asarray(c, dtype=float)
    out = np.full(len(c), np.nan)
    for t in range(1, len(c) - 1):
        if c[t] != 0:
            arg = (c[t - 1] + c[t + 1]) / (2.0 * c[t])
            if arg >= 1.0:
                out[t] = np.arccosh(arg)
    return out


def bootstrap_correlator(data: np.ndarray, nboot: int, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    ncfg = data.shape[0]
    central = periodic_correlator(data)
    if ncfg < 2 or nboot <= 1:
        return central, np.full_like(central, np.nan)
    boots = []
    for _ in range(nboot):
        idx = rng.integers(0, ncfg, size=ncfg)
        boots.append(periodic_correlator(data[idx]))
    return central, np.std(np.stack(boots), axis=0, ddof=1)


def integrated_autocorr_time(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n < 4:
        return float("nan")
    x = x - x.mean()
    var = np.dot(x, x) / n
    if var <= 0:
        return 0.5
    tau = 0.5
    for lag in range(1, min(n // 2, 1000)):
        rho = np.dot(x[:-lag], x[lag:]) / ((n - lag) * var)
        if rho <= 0:
            break
        tau += rho
    return float(tau)


def creutz_from_wilson(wilson_samples: List[Dict[str, float]]) -> Dict[str, float]:
    if not wilson_samples:
        return {}
    keys = sorted(wilson_samples[0].keys())
    means = {k: float(np.mean([row[k] for row in wilson_samples])) for k in keys}
    result: Dict[str, float] = {}
    for r in (1, 2):
        for t in (1, 2, 3):
            needed = (f"W{r}_{t}", f"W{r+1}_{t}", f"W{r}_{t+1}", f"W{r+1}_{t+1}")
            if all(k in means and means[k] > 0 for k in needed):
                ratio = means[f"W{r+1}_{t+1}"] * means[f"W{r}_{t}"] / (
                    means[f"W{r+1}_{t}"] * means[f"W{r}_{t+1}"]
                )
                if ratio > 0:
                    result[f"chi_{r}_{t}"] = float(-math.log(ratio))
    result.update({f"mean_{k}": v for k, v in means.items()})
    return result


def save_checkpoint(
    U: jax.Array,
    key: jax.Array,
    epsilon: float,
    sweep_count: int,
    plaquettes: List[float],
    acceptances: List[float],
    t1_ops: List[np.ndarray],
    torelons: List[np.ndarray],
    wilson_samples: List[Dict[str, float]],
) -> None:
    payload = {
        "U": np.asarray(jax.device_get(U)),
        "key": np.asarray(jax.device_get(key)),
        "epsilon": np.array(epsilon),
        "sweep_count": np.array(sweep_count),
        "plaquettes": np.asarray(plaquettes, dtype=float),
        "acceptances": np.asarray(acceptances, dtype=float),
        "t1_ops": np.asarray(t1_ops, dtype=np.complex128),
        "torelons": np.asarray(torelons, dtype=np.complex128),
        "wilson_json": np.array(json.dumps(wilson_samples)),
        "config_json": np.array(json.dumps(asdict(CFG))),
    }
    tmp = CHECKPOINT.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, **payload)
    tmp.replace(CHECKPOINT)
    print(f"checkpoint: {CHECKPOINT} ({len(plaquettes)} measurements)")


def load_checkpoint():
    if not CHECKPOINT.exists():
        return None
    data = np.load(CHECKPOINT, allow_pickle=False)
    old_cfg = json.loads(str(data["config_json"]))
    critical = ("L", "LT", "beta", "seed")
    for key in critical:
        if old_cfg.get(key) != asdict(CFG).get(key):
            print(f"Ignoring checkpoint: config mismatch in {key}.")
            return None
    return {
        "U": jnp.asarray(data["U"], dtype=CDTYPE),
        "key": jnp.asarray(data["key"]),
        "epsilon": float(data["epsilon"]),
        "sweep_count": int(data["sweep_count"]),
        "plaquettes": list(np.asarray(data["plaquettes"], dtype=float)),
        "acceptances": list(np.asarray(data["acceptances"], dtype=float)),
        "t1_ops": list(np.asarray(data["t1_ops"], dtype=np.complex128)),
        "torelons": list(np.asarray(data["torelons"], dtype=np.complex128)),
        "wilson_samples": json.loads(str(data["wilson_json"])),
    }


def make_plots(
    plaquettes: np.ndarray,
    acceptances: np.ndarray,
    t1_corr: Dict[str, np.ndarray],
    t1_err: Dict[str, np.ndarray],
    tor_corr: np.ndarray,
    tor_err: np.ndarray,
) -> None:
    plt.figure(figsize=(8, 4))
    plt.plot(plaquettes, lw=1)
    plt.xlabel("measurement")
    plt.ylabel("average plaquette")
    plt.tight_layout()
    plt.savefig(ROOT / "plaquette_history.png", dpi=180)
    plt.close()

    plt.figure(figsize=(8, 4))
    plt.plot(acceptances, lw=1)
    plt.axhline(CFG.target_accept, ls="--", lw=1)
    plt.xlabel("sweep")
    plt.ylabel("Metropolis acceptance")
    plt.tight_layout()
    plt.savefig(ROOT / "acceptance_history.png", dpi=180)
    plt.close()

    plt.figure(figsize=(8, 5))
    for name in MOMENTUM_NAMES:
        c = t1_corr[name]
        e = t1_err[name]
        t = np.arange(min(CFG.LT // 2 + 1, len(c)))
        denom = c[0] if c[0] != 0 else 1.0
        plt.errorbar(t, c[t] / denom, yerr=e[t] / abs(denom), marker="o", ms=3, capsize=2, label=name)
    plt.yscale("symlog", linthresh=1e-5)
    plt.xlabel(r"$\tau$")
    plt.ylabel(r"$C_{T_1^{+-}}(\tau)/C(0)$")
    plt.legend()
    plt.tight_layout()
    plt.savefig(ROOT / "t1pm_correlators.png", dpi=180)
    plt.close()

    plt.figure(figsize=(8, 5))
    t = np.arange(min(CFG.LT // 2 + 1, len(tor_corr)))
    denom = tor_corr[0] if tor_corr[0] != 0 else 1.0
    plt.errorbar(t, tor_corr[t] / denom, yerr=tor_err[t] / abs(denom), marker="o", ms=3, capsize=2)
    plt.yscale("symlog", linthresh=1e-6)
    plt.xlabel(r"$\tau$")
    plt.ylabel(r"$C_{\rm tor}(\tau)/C(0)$")
    plt.tight_layout()
    plt.savefig(ROOT / "torelon_correlator.png", dpi=180)
    plt.close()

    plt.figure(figsize=(8, 5))
    for name in MOMENTUM_NAMES:
        meff = effective_mass_cosh(t1_corr[name])
        t = np.arange(1, min(CFG.LT // 2, len(meff) - 1))
        plt.plot(t, meff[t], marker="o", ms=3, label=name)
    plt.xlabel(r"$\tau$")
    plt.ylabel("cosh effective energy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(ROOT / "t1pm_effective_energies.png", dpi=180)
    plt.close()


# -----------------------------------------------------------------------------
# Main run
# -----------------------------------------------------------------------------

def main() -> None:
    print("=" * 92)
    print("A100 SU(3) PURE-GAUGE + T1^{+-} MOMENTUM-SPECTROSCOPY PILOT")
    print("=" * 92)
    print("Config:", json.dumps(asdict(CFG), indent=2))
    print("Python:", platform.python_version())
    print("JAX:", jax.__version__)
    print("Devices:", jax.devices())
    print("Output:", ROOT)

    has_gpu = any(d.platform == "gpu" for d in jax.devices())
    if not has_gpu and not CFG.allow_cpu:
        raise RuntimeError(
            "No JAX GPU detected. In Colab select an A100 GPU runtime. "
            "For a tiny CPU smoke test only, set GLUE_ALLOW_CPU=1."
        )

    state = load_checkpoint()
    if state is None:
        key = random.PRNGKey(CFG.seed)
        key, kinit = random.split(key)
        if CFG.hot_start:
            print("Initializing Haar-random SU(3) gauge field on device...")
            U = haar_su3(kinit, CFG.shape4 + (4,))
        else:
            print("Initializing cold identity gauge field...")
            U = jnp.broadcast_to(I3, CFG.shape4 + (4, 3, 3))
        epsilon = CFG.epsilon
        sweep_count = 0
        plaquettes: List[float] = []
        acceptances: List[float] = []
        t1_ops: List[np.ndarray] = []
        torelons: List[np.ndarray] = []
        wilson_samples: List[Dict[str, float]] = []
    else:
        print(f"Resuming from {CHECKPOINT}")
        U = state["U"]
        key = state["key"]
        epsilon = state["epsilon"]
        sweep_count = state["sweep_count"]
        plaquettes = state["plaquettes"]
        acceptances = state["acceptances"]
        t1_ops = state["t1_ops"]
        torelons = state["torelons"]
        wilson_samples = state["wilson_samples"]

    # Compile representative kernels before timing production.
    print("Compiling update and measurement kernels...")
    t0 = time.time()
    U, key, acc0 = sweep(U, key, CFG.beta, epsilon)
    p0 = float(mean_plaquette(U))
    _ = np.asarray(jax.device_get(t1pm_momentum_operators(ape_smear(U, CFG.ape_alpha, CFG.ape_steps))))
    jax.block_until_ready(U)
    print(f"Compilation/warmup: {time.time()-t0:.1f} s; plaquette={p0:.8f}; acceptance={acc0:.3f}")

    # Thermalization only if there are no prior measurements.
    if len(plaquettes) == 0:
        print("\nThermalization")
        recent_acc: List[float] = []
        for s in range(CFG.therm_sweeps):
            U, key, acc = sweep(U, key, CFG.beta, epsilon)
            sweep_count += 1
            acceptances.append(acc)
            recent_acc.append(acc)
            if (s + 1) % CFG.adapt_every == 0:
                mean_acc = float(np.mean(recent_acc[-CFG.adapt_every:]))
                factor = math.exp(0.8 * (mean_acc - CFG.target_accept))
                epsilon = float(np.clip(epsilon * factor, 0.015, 1.2))
                p = float(mean_plaquette(U))
                print(
                    f"therm {s+1:4d}/{CFG.therm_sweeps}: "
                    f"P={p:.8f}, acc={mean_acc:.3f}, eps={epsilon:.4f}"
                )
        save_checkpoint(U, key, epsilon, sweep_count, plaquettes, acceptances, t1_ops, torelons, wilson_samples)

    print("\nProduction measurements")
    start_measurement = len(plaquettes)
    wall_start = time.time()
    for m in range(start_measurement, CFG.n_measurements):
        gap_acc = []
        for _ in range(CFG.sweeps_between):
            U, key, acc = sweep(U, key, CFG.beta, epsilon)
            sweep_count += 1
            acceptances.append(acc)
            gap_acc.append(acc)

        p = float(mean_plaquette(U))
        U_sm = ape_smear(U, CFG.ape_alpha, CFG.ape_steps)
        op = np.asarray(jax.device_get(t1pm_momentum_operators(U_sm)))
        tor = np.asarray(jax.device_get(spatial_torelon_operators(U_sm)))
        plaquettes.append(p)
        t1_ops.append(op)
        torelons.append(tor)

        if m % CFG.wilson_every == 0:
            row: Dict[str, float] = {}
            for r in (1, 2, 3):
                for t in (1, 2, 3, 4):
                    vals = [float(rectangular_loop(U_sm, mu, 0, r, t)) for mu in (1, 2, 3)]
                    row[f"W{r}_{t}"] = float(np.mean(vals))
            wilson_samples.append(row)

        elapsed = time.time() - wall_start
        rate = elapsed / max(1, m + 1 - start_measurement)
        print(
            f"meas {m+1:4d}/{CFG.n_measurements}: P={p:.8f}, "
            f"acc={np.mean(gap_acc):.3f}, eps={epsilon:.4f}, {rate:.2f} s/meas"
        )

        if (m + 1) % CFG.checkpoint_every == 0 or m + 1 == CFG.n_measurements:
            save_checkpoint(U, key, epsilon, sweep_count, plaquettes, acceptances, t1_ops, torelons, wilson_samples)

    plaquette_arr = np.asarray(plaquettes, dtype=float)
    acceptance_arr = np.asarray(acceptances, dtype=float)
    t1_arr = np.asarray(t1_ops, dtype=np.complex128)  # [cfg,k,t,c]
    tor_arr = np.asarray(torelons, dtype=np.complex128)  # [cfg,t,dir]

    t1_corr: Dict[str, np.ndarray] = {}
    t1_err: Dict[str, np.ndarray] = {}
    for ik, name in enumerate(MOMENTUM_NAMES):
        c, e = bootstrap_correlator(t1_arr[:, ik, :, :], CFG.bootstrap_samples, CFG.seed + 100 + ik)
        t1_corr[name] = c
        t1_err[name] = e
    tor_corr, tor_err = bootstrap_correlator(tor_arr, CFG.bootstrap_samples, CFG.seed + 200)

    creutz = creutz_from_wilson(wilson_samples)
    tau_int = integrated_autocorr_time(plaquette_arr)

    np.savez_compressed(
        RAW_NPZ,
        config_json=np.array(json.dumps(asdict(CFG))),
        plaquette=plaquette_arr,
        acceptance=acceptance_arr,
        t1_operators=t1_arr,
        torelon_operators=tor_arr,
        t1_corr=np.stack([t1_corr[n] for n in MOMENTUM_NAMES]),
        t1_corr_err=np.stack([t1_err[n] for n in MOMENTUM_NAMES]),
        torelon_corr=tor_corr,
        torelon_corr_err=tor_err,
        momentum_names=np.asarray(MOMENTUM_NAMES),
        wilson_json=np.array(json.dumps(wilson_samples)),
    )

    summary = {
        "title": "A100 SU(3) pure-gauge T1+- momentum spectroscopy pilot",
        "status": "COMPLETE",
        "scope": "isotropic Euclidean numerical pilot; not an exact certificate or Hamiltonian-limit coefficient test",
        "config": asdict(CFG),
        "device": [str(d) for d in jax.devices()],
        "measurements": len(plaquette_arr),
        "mean_plaquette": float(np.mean(plaquette_arr)),
        "stderr_plaquette_naive": float(np.std(plaquette_arr, ddof=1) / math.sqrt(max(1, len(plaquette_arr)))),
        "plaquette_tau_int_measurements": tau_int,
        "mean_acceptance": float(np.mean(acceptance_arr)),
        "final_epsilon": epsilon,
        "creutz_and_wilson": creutz,
        "t1_cosh_effective_energies": {
            n: [None if not np.isfinite(x) else float(x) for x in effective_mass_cosh(t1_corr[n])]
            for n in MOMENTUM_NAMES
        },
        "torelon_cosh_effective_energy": [
            None if not np.isfinite(x) else float(x) for x in effective_mass_cosh(tor_corr)
        ],
        "files": {
            "raw": str(RAW_NPZ),
            "checkpoint": str(CHECKPOINT),
            "log": str(LOG_TXT),
        },
    }
    SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    make_plots(plaquette_arr, acceptance_arr, t1_corr, t1_err, tor_corr, tor_err)

    print("\n" + "=" * 92)
    print("RUN COMPLETE")
    print("=" * 92)
    print(f"mean plaquette       = {summary['mean_plaquette']:.10f}")
    print(f"plaquette tau_int    = {tau_int:.3f} measurements")
    print(f"mean acceptance      = {summary['mean_acceptance']:.4f}")
    print("Creutz/string proxies:")
    for k, v in creutz.items():
        if k.startswith("chi_"):
            print(f"  {k:12s} = {v:+.8f}")
    print("\nOutputs:")
    for path in sorted(ROOT.iterdir()):
        print(" ", path)
    print("\nInterpretation rule: GPU results are numerical evidence. Exact strong-coupling")
    print("coefficients and theorem claims remain certified by the CPU rational pipeline.")


if __name__ == "__main__":
    main()
